# E0 · Exportar, quantizar e medir o twitter-XLM-R para servir

Parte dos pesos que o notebook 1 deixou em `Meu Drive/luciola/modelos/`.
Responde as três perguntas que decidem se ele pode substituir o stack na
Hetzner:

1. **Cabe?** Tamanho em disco de cada variante.
2. **Responde a tempo?** Latência em **CPU**, que é o que a Hetzner tem. O
   stack servido hoje responde em 1,5 ms.
3. **Passa no gate de viés?** A sonda de identidade é condição de release
   declarada em `methodology/pt_recall_x_vies.md`. Quantização muda score, então
   a sonda roda em cada variante, não só no original.

**Runtime: CPU.** Troque em Ambiente de execução, Alterar tipo. Rodar em GPU aqui
mede a máquina errada.

Leva uns 20 a 40 minutos.

## 1 · Confirmar que o runtime é CPU

In [ ]:
import torch
assert not torch.cuda.is_available(), (
    'Este notebook mede latencia de CPU, que e o que a Hetzner tem. '
    'Troque o runtime para CPU e rode de novo.')
!cat /proc/cpuinfo | grep 'model name' | head -2
!nproc

## 2 · Dependências, repositório e imports (célula única)

In [ ]:
!pip install -q 'transformers>=4.56' 'optimum[onnxruntime]>=1.20' 'onnxruntime>=1.18' \
    'scikit-learn>=1.4' 'pyarrow>=15' 'pyyaml>=6.0' 'lingua-language-detector>=2.0'

!git clone -q https://github.com/isasaade-23/hate-speech-nlp-en-pt.git /content/repo

import json, os, shutil, sys, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

REPO = Path('/content/repo')
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
os.chdir(REPO)

# as frases da sonda vem do repositorio: nao podem divergir das do gate local
from pt_identity_probe import FRASES, CONTROLES
print(f'sonda: {len(FRASES)} frases neutras, {len(CONTROLES)} controles de odio')

## 3 · Os pesos e o corpus, do Drive

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/luciola'
from google.colab import drive
drive.mount('/content/drive')

META = json.load(open(Path(DRIVE_DIR) / 'modelos' / 'twitter_xlmr_strict_s42' / 'luciola_serve.json'))
PESOS = Path(DRIVE_DIR) / 'modelos' / META['model_id']
MAXLEN, TEXT_COL = META['max_length'], META['text_column']
print('pesos:', PESOS, '| limiar do treino:', META['threshold'])

CORPUS = pd.read_parquet(Path(DRIVE_DIR) / 'corpus_strict.parquet')
VA = CORPUS[(CORPUS.split == 'val') & CORPUS.language.isin(['en', 'pt'])].reset_index(drop=True)
TE = CORPUS[(CORPUS.split == 'test') & CORPUS.language.isin(['en', 'pt'])].reset_index(drop=True)
print(f'val={len(VA)} test={len(TE)}')

## 4 · Exportar para ONNX e quantizar em int8

Quantização **dinâmica**, perfil AVX2: a Hetzner é AMD x86. Dinâmica e não
estática porque não exige conjunto de calibração e o erro fica bem menor em
modelo de sequência curta.

In [ ]:
ONNX_FP32 = Path('/content/onnx_fp32')
ONNX_INT8 = Path('/content/onnx_int8')

ort = ORTModelForSequenceClassification.from_pretrained(PESOS, export=True)
ort.save_pretrained(ONNX_FP32)
tok = AutoTokenizer.from_pretrained(PESOS)
tok.save_pretrained(ONNX_FP32)

quantizer = ORTQuantizer.from_pretrained(ONNX_FP32)
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer.quantize(save_dir=ONNX_INT8, quantization_config=qconfig)
tok.save_pretrained(ONNX_INT8)

def tamanho_mb(d):
    return round(sum(f.stat().st_size for f in Path(d).rglob('*') if f.is_file()) / 1e6, 1)

TAM = {'torch_fp32': tamanho_mb(PESOS), 'onnx_fp32': tamanho_mb(ONNX_FP32),
       'onnx_int8': tamanho_mb(ONNX_INT8)}
for k, v in TAM.items():
    print(f'{k:12s} {v:8.1f} MB')

## 5 · As três variantes atrás da mesma interface

In [ ]:
torch_model = AutoModelForSequenceClassification.from_pretrained(PESOS).eval()
ort_fp32 = ORTModelForSequenceClassification.from_pretrained(ONNX_FP32)
ort_int8 = ORTModelForSequenceClassification.from_pretrained(ONNX_INT8)

def scores(modelo, textos, batch=32):
    """Probabilidade da classe 1 para uma lista de textos."""
    saida = []
    for i in range(0, len(textos), batch):
        lote = [str(t) for t in textos[i:i+batch]]
        enc = tok(lote, truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
        with torch.no_grad():
            logits = modelo(**enc).logits
        logits = torch.as_tensor(logits).float()
        saida.append(torch.softmax(logits, 1).numpy()[:, 1])
    return np.concatenate(saida)

VARIANTES = {'torch_fp32': torch_model, 'onnx_fp32': ort_fp32, 'onnx_int8': ort_int8}
print('teste rapido:', {k: round(float(scores(m, ['eu te odeio, seu lixo'])[0]), 4)
                        for k, m in VARIANTES.items()})

## 6 · Latência em CPU

Duas medidas separadas, porque o produto usa as duas: **um texto por vez**, que
é a extensão e o site, e **lote de 32**, que é a análise de documento longo.

Referência do que roda hoje: 1,5 ms p50, um texto por vez.

In [ ]:
AMOSTRA = TE[TEXT_COL].astype(str).sample(80, random_state=42).tolist()

def mede(modelo, textos, batch, repeticoes=3):
    for t in textos[:5]:  # aquecimento: a primeira chamada carrega grafo e cache
        scores(modelo, [t], batch=1)
    tempos = []
    for _ in range(repeticoes):
        for i in range(0, len(textos), batch):
            lote = textos[i:i+batch]
            t0 = time.perf_counter()
            scores(modelo, lote, batch=batch)
            tempos.append((time.perf_counter() - t0) * 1000 / len(lote))
    return np.percentile(tempos, 50), np.percentile(tempos, 95)

linhas = []
for nome, modelo in VARIANTES.items():
    p50_1, p95_1 = mede(modelo, AMOSTRA, batch=1)
    p50_32, p95_32 = mede(modelo, AMOSTRA, batch=32)
    linhas.append({'variante': nome, 'tamanho_mb': TAM[nome],
                   'ms_p50_unitario': round(p50_1, 2), 'ms_p95_unitario': round(p95_1, 2),
                   'ms_p50_por_texto_lote32': round(p50_32, 2)})
    print(linhas[-1])

LATENCIA = pd.DataFrame(linhas)
print()
print(LATENCIA.to_string(index=False))
print('\nAviso: a CPU do Colab nao e a da Hetzner. Serve para ordenar as variantes,')
print('nao como numero final. O numero final se mede na maquina.')

## 7 · Qualidade de cada variante

O limiar é reajustado **na validação para cada variante**, porque quantizar
muda a escala do score. Reaproveitar o limiar do fp32 no int8 seria comparar
duas coisas em pontos de operação diferentes.

In [ ]:
def best_threshold(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score, dtype=float)
    cands = np.unique(np.quantile(y_score, np.linspace(0.02, 0.98, 97)))
    best_t, best_f = 0.5, -1.0
    for t in cands:
        f = f1_score(y_true, (y_score >= t).astype(int), average='macro', zero_division=0)
        if f > best_f: best_f, best_t = f, float(t)
    return best_t

SCORES, LIMIARES, qual = {}, {}, []
for nome, modelo in VARIANTES.items():
    s_va = scores(modelo, VA[TEXT_COL].tolist())
    thr = best_threshold(VA['label'].values, s_va)
    s_te = scores(modelo, TE[TEXT_COL].tolist())
    SCORES[nome], LIMIARES[nome] = s_te, thr
    for corte, mask in (('total', np.ones(len(TE), dtype=bool)),
                        ('en', (TE.language == 'en').values),
                        ('pt', (TE.language == 'pt').values)):
        y, sc_ = TE.loc[mask, 'label'].values, s_te[mask]
        yp = (sc_ >= thr).astype(int)
        qual.append({'variante': nome, 'limiar': round(thr, 4), 'corte': corte, 'n': len(y),
                     'macro_f1': round(f1_score(y, yp, average='macro', zero_division=0), 4),
                     'recall_hate': round(recall_score(y, yp, zero_division=0), 4),
                     'precision_hate': round(precision_score(y, yp, zero_division=0), 4)})
    print(nome, 'ok')

QUALIDADE = pd.DataFrame(qual)
print()
print(QUALIDADE.to_string(index=False))

# quantos rotulos o int8 troca em relacao ao fp32: o custo real da quantizacao
for a, b in (('onnx_fp32', 'torch_fp32'), ('onnx_int8', 'onnx_fp32')):
    ya = (SCORES[a] >= LIMIARES[a]).astype(int)
    yb = (SCORES[b] >= LIMIARES[b]).astype(int)
    print(f'{a} x {b}: concordancia {100*(ya == yb).mean():.2f}% ({int((ya != yb).sum())} rotulos diferentes)')

## 8 · O gate de viés de identidade

Vinte frases neutras ou positivas em português com termo de identidade. Nenhuma
é ódio, então toda marcação ali é falso positivo puro.

Referência a bater: o **stack servido hoje marca 3 de 20**. O modelo dedicado ao
PT marcava 9 de 20 e foi barrado. **Se alguma variante passar de 3, ela não vai
ao ar**, por melhor que seja o recall.

In [ ]:
from hsc.clean import clean_text
from hsc.config import data_config
PERFIL = data_config()['clean']['profiles']['light']

# o modelo foi treinado em text_clean: a sonda tem que passar pela mesma limpeza
neutras = [clean_text(f, PERFIL) for f in FRASES]
controles = [clean_text(f, PERFIL) for f in CONTROLES]

sonda = []
for nome, modelo in VARIANTES.items():
    thr = LIMIARES[nome]
    s_n, s_c = scores(modelo, neutras), scores(modelo, controles)
    fp = int((s_n >= thr).sum()); tp = int((s_c >= thr).sum())
    sonda.append({'variante': nome, 'limiar': round(thr, 4),
                  'fp_identidade': fp, 'n_sonda': len(FRASES),
                  'acertos_controle': tp, 'n_controle': len(CONTROLES),
                  'passa_no_gate': fp <= 3})
    print(f'{nome}: {fp}/{len(FRASES)} falsos positivos de identidade,'
          f' {tp}/{len(CONTROLES)} controles | {"PASSA" if fp <= 3 else "REPROVA"}')
    for f, s in zip(FRASES, s_n):
        if s >= thr:
            print(f'    {s:.3f}  {f}')

SONDA = pd.DataFrame(sonda)
print()
print(SONDA.to_string(index=False))

## 9 · Empacotar

As tabelas voltam por download. O ONNX int8 fica no Drive: é ele que sobe para
a Hetzner, se passar nos três testes.

In [ ]:
os.makedirs('/content/saida', exist_ok=True)
LATENCIA.to_csv('/content/saida/xlmr_latencia_cpu.csv', index=False)
QUALIDADE.to_csv('/content/saida/xlmr_qualidade_variantes.csv', index=False)
SONDA.to_csv('/content/saida/xlmr_sonda_identidade.csv', index=False)
json.dump({k: round(float(v), 4) for k, v in LIMIARES.items()},
          open('/content/saida/xlmr_limiares.json', 'w'), indent=2)

ALVO = '/content/xlmr_bench_resultado.zip'
with zipfile.ZipFile(ALVO, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in Path('/content/saida').iterdir():
        z.write(f, f'reports/tables/{f.name}')

shutil.copytree(ONNX_INT8, Path(DRIVE_DIR) / 'modelos' / f"{META['model_id']}_onnx_int8",
                dirs_exist_ok=True)
shutil.copy(ALVO, Path(DRIVE_DIR) / 'xlmr_bench_resultado.zip')
print('int8 no Drive:', Path(DRIVE_DIR) / 'modelos' / f"{META['model_id']}_onnx_int8")

try:
    from google.colab import files
    files.download(ALVO)
except ImportError:
    print('rodando local:', ALVO)

---

**Me mande:** o `xlmr_bench_resultado.zip` e a saída das células 6, 7 e 8.

Com isso eu decido três coisas de uma vez: se serve int8 ou fp32, qual limiar
vai para o registry, e se o modelo pode mesmo substituir o stack. Se a célula 8
reprovar todas as variantes, o twitter-XLM-R cai como o modelo dedicado caiu, e
o caminho passa a ser só o corpus sintético.